In [ ]:
# 1. УСТАНОВКА И АВТОРИЗАЦИЯ

!pip install -q kaggle

import os, zipfile, shutil, re
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.model_selection import train_test_split
from google.colab import files

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

# Авторизация Kaggle
if not os.path.exists('/root/.kaggle/kaggle.json'):
    print("Загрузите файл kaggle.json")
    uploaded = files.upload()
    os.makedirs('/root/.kaggle', exist_ok=True)
    for fname in uploaded.keys():
        if fname == 'kaggle.json':
            shutil.move(fname, '/root/.kaggle/kaggle.json')
            os.chmod('/root/.kaggle/kaggle.json', 0o600)
            break

TensorFlow: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Загрузите файл kaggle.json


Saving kaggle.json to kaggle.json


In [ ]:
!kaggle competitions list

ref                                                                           deadline             category         reward  teamCount  userHasEntered  
----------------------------------------------------------------------------  -------------------  --------  -------------  ---------  --------------  
https://www.kaggle.com/competitions/passenger-screening-algorithm-challenge   2017-12-15 23:59:00  Featured  1,500,000 Usd        518           False  
https://www.kaggle.com/competitions/zillow-prize-1                            2018-01-10 15:59:00  Featured  1,200,000 Usd       3770           False  
https://www.kaggle.com/competitions/data-science-bowl-2017                    2017-04-12 23:59:00  Featured  1,000,000 Usd       1972           False  
https://www.kaggle.com/competitions/vesuvius-challenge-ink-detection          2023-06-14 23:59:00  Featured  1,000,000 Usd       1249           False  
https://www.kaggle.com/competitions/arc-prize-2026-arc-agi-3                  2026-11-02

In [ ]:
# 2. СКАЧИВАНИЕ ДАТАСЕТА
!kaggle competitions download -c the-nature-conservancy-fisheries-monitoring -p /content/data
!yes | unzip -q /content/data/the-nature-conservancy-fisheries-monitoring.zip -d /content/data
!yes | unzip -q /content/data/train.zip -d /content/data
!yes | unzip -q /content/data/test_stg1.zip -d /content/data
!rm -f /content/data/*.zip

TRAIN_DIR = '/content/data/train'
TEST_DIR  = '/content/data/test_stg1'

print("Train files:", len(os.listdir(TRAIN_DIR)))
print("Test files:", len(os.listdir(TEST_DIR)))

100% 2.11G/2.11G [02:02<00:00, 18.5MB/s]

Train files: 9
Test files: 1000


In [ ]:
!ls /content/data/train

ALB  BET  DOL  LAG  NoF  OTHER	SHARK  YFT


In [ ]:
# ЗАГРУЗКА ДАННЫХ (с подпапками)


IMG_SIZE = (299, 299)
BATCH_SIZE = 32
SEED = 42
EPOCHS1, EPOCHS2 = 10, 10
NUM_CLASSES = 8


import os
import numpy as np
from sklearn.model_selection import train_test_split

# Словарь классов (убедитесь, что определён ранее)
CLASSES = {
    'ALB': 0, 'BET': 1, 'DOL': 2, 'LAG': 3,
    'NoF': 4, 'OTHER': 5, 'SHARK': 6, 'YFT': 7
}

TRAIN_DIR = '/content/data/train'
TEST_DIR  = '/content/data/test_stg1'

def get_files_labels(dir_path):
    files, labels = [], []
    for class_name in os.listdir(dir_path):
        class_path = os.path.join(dir_path, class_name)
        if not os.path.isdir(class_path):
            continue
        if class_name not in CLASSES:
            continue
        label = CLASSES[class_name]
        for fname in os.listdir(class_path):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                files.append(os.path.join(class_path, fname))
                labels.append(label)
    return files, labels

train_files, train_labels = get_files_labels(TRAIN_DIR)
test_files = sorted([os.path.join(TEST_DIR, f) for f in os.listdir(TEST_DIR)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

print(f"Train: {len(train_files)}, Test: {len(test_files)}")

Train: 3777, Test: 1000


In [ ]:
# Разбивка на train/val (80/20)
train_files, val_files, train_labels, val_labels = train_test_split(
    train_files, train_labels, test_size=0.2, random_state=SEED, stratify=train_labels
)

print(f"Train: {len(train_files)}, Val: {len(val_files)}")

Train: 3021, Val: 756


In [ ]:
# 4. ГЕНЕРАТОРЫ С АУГМЕНТАЦИЕЙ
preprocess = tf.keras.applications.inception_v3.preprocess_input

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(preprocessing_function=preprocess)

def create_generator(files, labels, datagen, batch_size, shuffle=True):
    while True:
        indices = np.arange(len(files))
        if shuffle:
            np.random.shuffle(indices)
        for start in range(0, len(files), batch_size):
            end = min(start + batch_size, len(files))
            batch_indices = indices[start:end]
            batch_files = [files[i] for i in batch_indices]
            batch_labels = [labels[i] for i in batch_indices]
            batch_images = []
            for fpath in batch_files:
                img = tf.keras.utils.load_img(fpath, target_size=IMG_SIZE)
                img = tf.keras.utils.img_to_array(img)
                batch_images.append(img)
            batch_images = np.array(batch_images)
            batch_labels = tf.keras.utils.to_categorical(batch_labels, NUM_CLASSES)
            # Применяю аугментацию
            augmented = datagen.flow(batch_images, batch_labels, batch_size=len(batch_images), shuffle=False)
            x_batch, y_batch = next(augmented)
            yield x_batch, y_batch

train_gen = create_generator(train_files, train_labels, train_datagen, BATCH_SIZE, shuffle=True)
val_gen   = create_generator(val_files,   val_labels,   val_datagen,   BATCH_SIZE, shuffle=False)

steps_per_epoch = max(1, len(train_files) // BATCH_SIZE)
val_steps = max(1, len(val_files) // BATCH_SIZE)

In [ ]:
# 5. МОДЕЛЬ (InceptionV3)
base = InceptionV3(weights='imagenet', include_top=False, input_shape=(*IMG_SIZE, 3))
base.trainable = False

x = base.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(1024, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
x = layers.Dropout(0.5)(x)
output = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = models.Model(inputs=base.input, outputs=output)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
              loss='categorical_crossentropy',
              metrics=['accuracy'])
model.summary()


87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 299, 299,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 149, 149,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 149, 149,  │         96 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 149, 149,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 147, 147,  │      9,216 │ activation[0][0]  │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 147, 147,  │         96 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 147, 147,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 147, 147,  │     18,432 │ activation_1[0][… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 147, 147,  │        192 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 147, 147,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 73, 73,    │          0 │ activation_2[0][… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 73, 73,    │      5,120 │ max_pooling2d[0]… │
│                     │ 80)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 73, 73,    │        240 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 80)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 73, 73,    │          0 │ batch_normalizat… │
│ (Activation)        │ 80)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 71, 71,    │    138,240 │ activation_3[0][… │
│                     │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 71, 71,    │        576 │ conv2d_4[0][0]    │
│ (BatchNormalizatio… │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_4        │ (None, 71, 71,    │          0 │ batch_normalizat

 Total params: 23,909,160 (91.21 MB)

 Trainable params: 2,106,376 (8.04 MB)

 Non-trainable params: 21,802,784 (83.17 MB)

In [ ]:
# 6. КОЛБЭКИ
callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(factor=0.2, patience=3, min_lr=1e-6, verbose=1),
    ModelCheckpoint('best.h5', save_best_only=True, verbose=1)
]

In [ ]:
# 7. ОБУЧЕНИЕ

print("=== Этап 1: обучение головы ===")
model.fit(train_gen, steps_per_epoch=steps_per_epoch, epochs=EPOCHS1,
          validation_data=val_gen, validation_steps=val_steps,
          callbacks=callbacks, verbose=1)

print("=== Этап 2: тонкая настройка ===")
base.trainable = True
for layer in base.layers[:100]:
    layer.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
              loss='categorical_crossentropy',
              metrics=['accuracy'])
model.fit(train_gen, steps_per_epoch=steps_per_epoch, epochs=EPOCHS2,
          validation_data=val_gen, validation_steps=val_steps,
          callbacks=callbacks, verbose=1)

model.load_weights('best.h5')


=== Этап 1: обучение головы ===
Epoch 1/10
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - accuracy: 0.4066 - loss: 2.3725
Epoch 1: val_loss improved from None to 1.31620, saving model to best.h5



Epoch 1: finished saving model to best.h5
94/94 ━━━━━━━━━━━━━━━━━━━━ 961s 10s/step - accuracy: 0.4651 - loss: 1.8668 - val_accuracy: 0.5611 - val_loss: 1.3162 - learning_rate: 0.0010
Epoch 2/10
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - accuracy: 0.5470 - loss: 1.3792
Epoch 2: val_loss improved from 1.31620 to 1.20624, saving model to best.h5



Epoch 2: finished saving model to best.h5
94/94 ━━━━━━━━━━━━━━━━━━━━ 974s 10s/step - accuracy: 0.5507 - loss: 1.3915 - val_accuracy: 0.6087 - val_loss: 1.2062 - learning_rate: 0.0010
Epoch 3/10
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - accuracy: 0.5579 - loss: 1.3560
Epoch 3: val_loss improved from 1.20624 to 1.06632, saving model to best.h5



Epoch 3: finished saving model to best.h5
94/94 ━━━━━━━━━━━━━━━━━━━━ 923s 10s/step - accuracy: 0.5771 - loss: 1.2892 - val_accuracy: 0.6644 - val_loss: 1.0663 - learning_rate: 0.0010
Epoch 4/10
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - accuracy: 0.5812 - loss: 1.2241
Epoch 4: val_loss did not improve from 1.06632
94/94 ━━━━━━━━━━━━━━━━━━━━ 897s 10s/step - accuracy: 0.5942 - loss: 1.1962 - val_accuracy: 0.6257 - val_loss: 1.1089 - learning_rate: 0.0010
Epoch 5/10
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - accuracy: 0.6221 - loss: 1.1509
Epoch 5: val_loss improved from 1.06632 to 0.99270, saving model to best.h5



Epoch 5: finished saving model to best.h5
94/94 ━━━━━━━━━━━━━━━━━━━━ 922s 10s/step - accuracy: 0.6163 - loss: 1.1612 - val_accuracy: 0.6699 - val_loss: 0.9927 - learning_rate: 0.0010
Epoch 6/10
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - accuracy: 0.6413 - loss: 1.0921
Epoch 6: val_loss did not improve from 0.99270
94/94 ━━━━━━━━━━━━━━━━━━━━ 920s 10s/step - accuracy: 0.6296 - loss: 1.1113 - val_accuracy: 0.6975 - val_loss: 1.0214 - learning_rate: 0.0010
Epoch 7/10
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - accuracy: 0.6436 - loss: 1.0683
Epoch 7: val_loss improved from 0.99270 to 0.93218, saving model to best.h5



Epoch 7: finished saving model to best.h5
94/94 ━━━━━━━━━━━━━━━━━━━━ 883s 9s/step - accuracy: 0.6457 - loss: 1.0628 - val_accuracy: 0.7155 - val_loss: 0.9322 - learning_rate: 0.0010
Epoch 8/10
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - accuracy: 0.6222 - loss: 1.0889
Epoch 8: val_loss improved from 0.93218 to 0.89777, saving model to best.h5



Epoch 8: finished saving model to best.h5
94/94 ━━━━━━━━━━━━━━━━━━━━ 916s 10s/step - accuracy: 0.6420 - loss: 1.0616 - val_accuracy: 0.6961 - val_loss: 0.8978 - learning_rate: 0.0010
Epoch 9/10
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - accuracy: 0.6646 - loss: 1.0195
Epoch 9: val_loss did not improve from 0.89777
94/94 ━━━━━━━━━━━━━━━━━━━━ 889s 9s/step - accuracy: 0.6594 - loss: 1.0262 - val_accuracy: 0.6809 - val_loss: 0.9483 - learning_rate: 0.0010
Epoch 10/10
92/94 ━━━━━━━━━━━━━━━━━━━━ 15s 8s/step - accuracy: 0.6756 - loss: 0.9644

In [ ]:
# 8. ОЦЕНКА
val_loss, val_acc = model.evaluate(val_gen, steps=val_steps, verbose=0)
print(f"✅ val_loss = {val_loss:.6f}, val_acc = {val_acc:.4f}")

тестирование эпохи мвааооопаааапааапппаорповлоолааппп

In [ ]:
# 9. ПРЕДСКАЗАНИЯ НА ТЕСТЕ
test_datagen = ImageDataGenerator(preprocessing_function=preprocess)

def test_generator(files, batch_size):
    for i in range(0, len(files), batch_size):
        batch_files = files[i:i+batch_size]
        batch_images = []
        for fpath in batch_files:
            img = tf.keras.utils.load_img(fpath, target_size=IMG_SIZE)
            img = tf.keras.utils.img_to_array(img)
            batch_images.append(img)
        batch_images = np.array(batch_images)
        batch_images = test_datagen.standardize(batch_images)
        yield batch_images

test_steps = len(test_files) // BATCH_SIZE + 1
preds = []
for batch in test_generator(test_files, BATCH_SIZE):
    preds.extend(model.predict(batch, verbose=0))
    if len(preds) >= len(test_files):
        break
preds = np.array(preds[:len(test_files)])

In [ ]:


# ================================================================
# 10. СОЗДАНИЕ submission.csv
# ================================================================
# Для этого соревнования в submission.csv нужно указать вероятности для каждого класса
# Формат: image,ALB,BET,DOL,LAG,NoF,OTHER,SHARK,YFT
with open('submission.csv', 'w') as f:
    f.write('image,ALB,BET,DOL,LAG,NoF,OTHER,SHARK,YFT\n')
    for fname, probs in zip(test_files, preds):
        img_name = os.path.basename(fname)
        probs_str = ','.join([f'{p:.6f}' for p in probs])
        f.write(f'{img_name},{probs_str}\n')

print("✅ submission.csv создан. Скачиваю...")
files.download('submission.csv')